# B5T2 · Cuaderno de mando de la evaluación

Este cuaderno **conduce**; el código **vive en el paquete `agente/`**. La separación no es cosmética:

- El día 24 un evaluador clona el repo y ejecuta `evaluar("holdout.jsonl")` sin abrir ningún notebook. Si la lógica estuviera aquí, no funcionaría. Ese fue exactamente el problema del repo del 17-sep.
- Todo lo que se ve aquí se puede volver a generar con `python -m agente.cli …`. Este cuaderno es la misma máquina con ventanas.

Qué hace cada bloque:

| Bloque | Gasta API | Para qué |
|---|---|---|
| 1 · Preparación y **calentamiento** | no | comprobar corpus y clave, precargar el retrieval |
| 2 · Una pregunta | sí (1) | ver una trayectoria completa antes de lanzar 60 |
| 3 · Ejecutar una arquitectura | sí (20 × reps) | guarda un JSON por pregunta; **idempotente** |
| 4 · Leer la tabla | no | qué falla, y por qué, mirando la trayectoria guardada |
| 5 · Comparativa | no | la tabla del informe |
| 6 · Recall del retrieval | casi no | la tabla del §4.4 |

Cambia `ARQ` y vuelve a ejecutar los bloques 3-5 para cada peldaño:

`baseline` → `a1_guardrails` → `a2_retrieval` → `a3_hibrido` → `a4_comparativas` → `a5_limites` → `a6_cifras_texto` (`final`)

Cada peldaño **añade** al anterior y no quita nada, así la diferencia entre dos filas consecutivas se atribuye a una sola cosa.

## 1 · Preparación

**Por qué hay un calentamiento.** La primera vez que el agente llama a `search_filings`, Python carga el índice FAISS y el modelo de embeddings desde disco: entre 15 y 20 segundos. Si eso ocurre *dentro* de una pregunta, esos segundos se suman a su latencia y la columna del informe queda contaminada.

Y hay algo peor: el proceso del kernel sobrevive entre repeticiones, así que **solo la repetición 1 paga la carga**. En la tabla eso aparece como varianza entre repeticiones — precisamente lo que las tres repeticiones intentan medir. Un artefacto de arranque disfrazado de ruido del modelo.

`calentar()` lo saca fuera del cronómetro. `ejecutar()` la llama sola, pero conviene verla aquí para saber qué está pasando.

De paso, si el modelo ya está en la caché local, pone `HF_HUB_OFFLINE=1`: desaparece el aviso de peticiones anónimas a Hugging Face, se ahorra un viaje de red por arranque y —lo que de verdad importa— **el día 24 la ejecución no depende de que Hugging Face esté disponible**. En un clon recién hecho, sin caché, no se activa, para que la primera descarga funcione con normalidad.

In [1]:
# 1 · Preparación ---------------------------------------------------------------
# autoreload: si editas un fichero de agente/, el cuaderno lo recoge sin reiniciar.
%load_ext autoreload
%autoreload 2

import os, json, warnings
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# tqdm quiere dibujar una barra de progreso interactiva y necesita `ipywidgets`.
# Sin él se cae a la barra de texto, que funciona igual. Silenciamos el aviso.
# (Si prefieres las barras bonitas: `uv add --dev ipywidgets` y quita esta línea.)
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

from agente.config import ARQUITECTURAS, MODELO, REPETICIONES
from agente.corpus import dir_corpus, cargar_xbrl, raiz_repo
from agente.interfaz import (responder, ejecutar, puntuar, comparar, resumir, calentar,
                             dir_rep, dir_arquitectura, dir_resultados, leer_golden)
from agente.resultado import pretty_trace

# --- lo que vamos a evaluar en esta pasada ---
ARQ    = "baseline"                 # baseline · a1_guardrails · a2_retrieval · a3_hibrido · a4_comparativas · a5_limites · a6_cifras_texto
REPS   = REPETICIONES               # 3
GOLDEN = "data/golden_set.jsonl"

# --- comprobaciones, sin gastar nada ---
print("raíz del repo :", raiz_repo())
print("corpus        :", dir_corpus(), "·", len(cargar_xbrl()), "hechos XBRL")
print("modelo        :", MODELO)
print("clave OpenRouter en el entorno:", bool(os.environ.get("OPENROUTER_API_KEY")))   # solo sí/no, nunca el valor
HAY_CLAVE = bool(os.environ.get("OPENROUTER_API_KEY"))

golden = leer_golden(GOLDEN)
print(f"golden set    : {len(golden)} preguntas · "
      f"{sum(1 for g in golden if g.get('ancla_texto'))} con ancla")

# --- calentamiento: fuera del cronómetro, una sola vez por kernel ---
print("\ncalentando el retrieval…")
calentar()
print("HF_HUB_OFFLINE =", os.environ.get("HF_HUB_OFFLINE", "(sin poner)"))

# Las arquitecturas, como tabla: qué enciende cada una
display(pd.DataFrame([a.como_dict() for n, a in ARQUITECTURAS.items() if n != "final"])
        .set_index("nombre")[["limites","verificador_cifras","verificador_cita","esquema_estricto",
                              "prompt","filtros_forzados","reescritura","hibrido","descripcion"]])

raíz del repo : C:\dev\MIAX_Tareas\B5T2-agentes
corpus        : C:\dev\MIAX_Tareas\B5T2-agentes\corpus · 135 hechos XBRL
modelo        : openrouter:google/gemini-3.8-flash
clave OpenRouter en el entorno: True
golden set    : 20 preguntas · 14 con ancla

calentando el retrieval…


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3250.52it/s]

  retrieval precargado: 1749 vectores, 1749 fragmentos (5.1 s)
HF_HUB_OFFLINE = 1


,limites,verificador_cifras,verificador_cita,esquema_estricto,prompt,filtros_forzados,reescritura,hibrido,descripcion
nombre,,,,,,,,,
baseline,False,False,False,False,base,False,False,False,"El agente del día 10, tal cual. Congelado antes de tocar..."
a1_guardrails,True,True,True,True,honesto,False,False,False,"+ límites de llamadas, verificador XBRL, verificador de ..."
a2_retrieval,True,True,True,True,honesto,True,True,False,+ filtros por metadatos forzados y reescritura de la con...
a3_hibrido,True,True,True,True,honesto,True,True,True,"+ híbrido BM25 con RRF, medido DESPUÉS de la reescritura."
a4_comparativas,True,True,True,True,comparativas,True,True,True,+ procedimiento explícito de comparativas y conceptos po...


### Limpiar antes de empezar

`ejecutar()` es **idempotente**: si ya existe `crudo/<id>.json`, salta la pregunta. Eso permite relanzar una ejecución cortada sin repetir lo hecho — pero también significa que **una medición mala se queda guardada** hasta que la borres.

- `BORRAR_TODO = True` deja la arquitectura a cero (crudo, tablas y resumen).
- `BORRAR_REP = n` borra solo esa repetición.
- Con los dos apagados, la celda **solo informa** de lo que hay.

Acuérdate de volver a dejarlos apagados después de borrar, o la próxima vez que pases por aquí te llevas por delante lo bueno.

In [ ]:
BORRAR_REP  = None           # p. ej. 1 para rehacer solo la repetición 1 de ARQ
BORRAR_TODO = False          # True para dejar ARQ completamente a cero

import shutil

if BORRAR_TODO:
    carpeta = dir_arquitectura(ARQ)
    if carpeta.is_dir():
        shutil.rmtree(carpeta)
    print(f"borrada entera: {carpeta}")
elif BORRAR_REP is not None:
    carpeta = dir_rep(ARQ, BORRAR_REP)
    if carpeta.is_dir():
        shutil.rmtree(carpeta)
    print(f"borrada: {carpeta}")
else:
    for carpeta in sorted(dir_arquitectura(ARQ).glob("rep*")):
        n = len(list((carpeta / "crudo").glob("*.json"))) if (carpeta / "crudo").is_dir() else 0
        print(f"{carpeta.name}: {n} preguntas guardadas")
    print("(nada borrado)")

## 2 · Una pregunta suelta

Antes de lanzar 60 llamadas, una. Se ve la trayectoria entera: qué herramienta pidió, con qué argumentos, qué le devolvió, y la respuesta estructurada. Es la misma `pretty_trace` de clase.

In [ ]:
PREGUNTA = "¿Cuál fue el revenue de NVIDIA en FY2025?"

if HAY_CLAVE:
    r = responder(PREGUNTA, thread_id="suelta", arquitectura=ARQ)
    print(pretty_trace(r))
    print(f"\n  [{r['latencia_s']:.1f} s · {r['coste_usd']*100:.2f} ¢]")
else:
    print("Sin clave: este bloque no se ejecuta.")

## 3 · Ejecutar una arquitectura, `REPS` veces

`ejecutar()` guarda **un JSON por pregunta** en `resultados/agente/<ARQ>/rep<n>/crudo/` con la trayectoria completa. Si esto se corta a mitad, vuelve a ejecutar la celda y sigue donde estaba.

`puntuar()` no gasta API: lee esos JSON, aplica los tres evaluadores y escribe `tabla.csv`. Como está separado, si mañana corregimos un evaluador se re-puntúa todo en segundos.

El `thread_id` lleva la repetición dentro (`baseline-rep2-gX-013`): sin eso, la repetición 2 vería la conversación de la 1 en el checkpointer.

In [ ]:
tablas = {}
if HAY_CLAVE:
    for rep in range(1, REPS + 1):
        print(f"\n== {ARQ} · repetición {rep}/{REPS} ==")
        ejecutar(GOLDEN, ARQ, rep)                 # API · idempotente · calienta antes del bucle
        tablas[rep] = puntuar(ARQ, rep)            # sin API · incluye recall@5
        display(resumir(tablas[rep], f"{ARQ} rep{rep}"))
else:
    # Sin clave se puede puntuar lo que ya esté guardado de otras veces.
    for carpeta in sorted(dir_arquitectura(ARQ).glob("rep*")):
        if (carpeta / "crudo").is_dir() and any((carpeta / "crudo").glob("*.json")):
            tablas[int(carpeta.name[3:])] = puntuar(ARQ, int(carpeta.name[3:]), con_recall=False)
    print("Sin clave: puntuado lo guardado →", list(tablas) or "nada")

## 4 · Leer la tabla: qué falla y por qué

Una fila por pregunta. Las columnas que importan:

- **`acierto`**: todos los evaluadores aplicables en `True`.
- **`cita` / `cifra` / `trayectoria`**: los tres del enunciado. Vacío = no aplica a esa familia.
- **`herramientas`**: el camino que siguió. Una numérica sin `get_xbrl_fact` suspende trayectoria aunque la cifra sea correcta.
- **`fuente`**: si dice `ninguna` en una pregunta con respuesta, no la encontró; si dice `xbrl` con una cifra que no cuadra, la leyó de donde no debía o la inventó.
- **`recall5` / `pos_ancla`**: el retriever, con los filtros del golden. `pos_ancla=6` y `pos_ancla=900` fallan igual el recall@5 y no son el mismo problema.

In [ ]:
REP_A_MIRAR = 1
if REP_A_MIRAR in tablas:
    t = tablas[REP_A_MIRAR]

    # `cifra`/`cita`/`trayectoria` son los VEREDICTOS; `cifra_dada`/`cita_dada`,
    # lo que respondió el agente. Ver los dos juntos es lo que delata si un fallo
    # es de capacidad o de convención (p. ej. poner la variación donde se espera
    # el valor del ejercicio).
    cols = ["id","familia","acierto","cita","cifra","trayectoria","fuente",
            "cifra_dada","cifra_esperada","ratio_cifra","herramientas",
            "n_llamadas","coste_usd","latencia_s","recall5","pos_ancla"]
    cols = [c for c in cols if c in t]

    def colorear(fila):
        return ["background-color:#fde2e2" if fila.get("acierto") is False else
                ("background-color:#e2f5e2" if fila.get("acierto") is True else "") for _ in fila]

    display(t[cols].style.apply(colorear, axis=1)
            .format({"coste_usd": "{:.4f}", "latencia_s": "{:.1f}",
                     "cifra_dada": "{:,.0f}", "cifra_esperada": "{:,.0f}",
                     "ratio_cifra": "{:.3f}"}, na_rep="—"))

    fallos = t[t["acierto"] == False]                     # noqa: E712
    print(f"\n{len(fallos)} fallos de {len(t)}. Por familia:")
    display(t.groupby("familia")["acierto"].agg(["mean", "count"]).rename(columns={"mean": "tasa"}))

    # Instrumentación: qué guardrail actuó. Un guardrail que nunca salta no ha
    # aportado nada, y eso se cuenta en vez de estimarse con una ablación.
    instr = [c for c in ["corrigio_cifra","corrigio_cita","reintentos_esquema",
                         "limite_alcanzado","n_busquedas","busquedas_con_ticker",
                         "busquedas_con_item","uso_read_section"] if c in t]
    if instr:
        print("\nGuardrails y retrieval (media sobre las 20 preguntas):")
        display(t[instr].mean().to_frame("valor").T)

    print("\nLas 3 más lentas y las 3 más caras:")
    display(t.nlargest(3, "latencia_s")[["id","latencia_s","n_llamadas","herramientas"]])
    display(t.nlargest(3, "coste_usd")[["id","coste_usd","n_llamadas","herramientas"]])
else:
    print("No hay tabla para esa repetición.")

### Las trayectorias de los fallos

Esto es lo que en clase había que imprimir a mano. Aquí se lee de lo guardado: **no gasta nada**. Para cada fallo, la pregunta, lo esperado, la trayectoria y la respuesta.

In [ ]:
def ver_traza(arq, rep, id_):
    reg = json.loads((dir_rep(arq, rep) / "crudo" / f"{id_}.json").read_text(encoding="utf-8"))
    it = reg["item"]
    print("=" * 88)
    print(f"[{it['id']} · {it['familia']}] {it['pregunta']}")
    print(f"  esperado: {str(it.get('respuesta_esperada'))[:110]}")
    if it.get("herramienta_esperada"):
        print(f"  herramientas esperadas: {it['herramienta_esperada']}")
    print()
    print(reg.get("error") or pretty_trace(reg["resultado"]))

if REP_A_MIRAR in tablas:
    for id_ in tablas[REP_A_MIRAR].loc[tablas[REP_A_MIRAR]["acierto"] == False, "id"]:    # noqa: E712
        ver_traza(ARQ, REP_A_MIRAR, id_)

# Y cualquier otra, aunque haya acertado:
#   ver_traza(ARQ, 1, "gX-003")

## 5 · La comparativa: una fila por arquitectura

Media de las repeticiones, con mínimo y máximo para ver si un salto es mejora o ruido. `comparativa.md` es la tabla del informe con el mejor valor de cada columna en negrita.

In [ ]:
comp = comparar()
cols = ["arquitectura","reps","acierto","acierto numerica","acierto extractiva","acierto comparativa",
        "cita","cifra","trayectoria","recall@5","coste medio (¢)","latencia media (s)","llamadas/pregunta"]
display(comp[[c for c in cols if c in comp]].round(3))

if len(comp) and "acierto min" in comp:
    print("\nRango entre repeticiones (acierto):")
    display(comp[["arquitectura","acierto min","acierto","acierto max"]].round(3))

md_path = dir_resultados() / "comparativa.md"
if md_path.is_file():
    display(Markdown(md_path.read_text(encoding="utf-8")))

## 6 · El retrieval solo: recall@5 por configuración

La tabla del §4.4. No interviene el agente: se mide el buscador con los filtros del golden set. Cero llamadas de API salvo la reescritura de la consulta (una por pregunta, cacheada en `resultados/retrieval/reescrituras.json`).

`posiciones` dice en qué puesto quedó el ancla en cada configuración: es lo que explica **por qué** una configuración gana.

In [ ]:
from agente.recall import medir_recall, CONFIGS

configs = dict(CONFIGS)
if not HAY_CLAVE:                       # sin clave no se puede reescribir
    configs = {k: v for k, v in configs.items() if "reescritura" not in k}

recall = medir_recall(configs)
display(recall)

pos = pd.read_csv(dir_resultados() / "retrieval" / "posiciones.csv")
display(pos.pivot(index="id", columns="config", values="pos_ancla"))

### 6b · El recall con las consultas REALES del agente

La tabla de arriba mide el retriever con la **pregunta del golden en español**. Pero el agente no manda esa pregunta: manda **su propia consulta**, que ya suele ir en inglés y con filtros (lo dice el prompt base). Así que «reescritura +14 pp» es una propiedad del retriever, no necesariamente del agente.

Esta celda coge todas las llamadas a `search_filings` que hizo una arquitectura (de sus trazas guardadas) y pregunta: *con lo que el agente buscó de verdad, ¿habría visto el ancla en el top-5 con cada retriever?* Una pregunta cuenta si **alguna** de sus búsquedas lo consigue. No gasta API salvo la reescritura (cacheada).

`mejor_pos_ancla` en `posiciones_trazas_<arq>.csv` dice a qué distancia se quedó cada pregunta.


In [ ]:
from agente.recall import recall_de_trazas, CONFIGS_AGENTE

ARQ_TRAZAS = "a1_guardrails"       # de qué arquitectura se leen las consultas
configs_agente = dict(CONFIGS_AGENTE)
if not HAY_CLAVE:
    configs_agente = {k: v for k, v in configs_agente.items() if "reescritura" not in k}

recall_trazas = recall_de_trazas(ARQ_TRAZAS, configs=configs_agente)
display(recall_trazas)

pos_t = pd.read_csv(dir_resultados() / "retrieval" / f"posiciones_trazas_{ARQ_TRAZAS}.csv")
display(pos_t.pivot(index="id", columns="config", values="mejor_pos_ancla"))


## 7 · El golden set adversario: honestidad, varias compañías, compañías × ejercicios

El golden original está agotado como instrumento: A4 y A6 alcanzan el 100 % de acierto medio. Y tiene un agujero: **ninguna pregunta sin respuesta**, así que el guardrail de honestidad (`fuente='ninguna'`) no se ha ejercitado nunca.

`data/golden_set_dificil.jsonl` son 15 preguntas en tres familias nuevas:

- **honestidad** (5): el dato no está en el corpus. Lo correcto es `fuente='ninguna'`. Cada una lleva una trampa (la cifra SÍ aparece en una tabla del Item 8, o se puede calcular desde la prosa).
- **multi** (5): dos o más compañías, un ejercicio. Convención: si se pregunta *quién*, `ticker`=ganadora y `cifra`=su valor; si se pregunta una diferencia o ratio, `cifra`=esa magnitud.
- **multi_temporal** (5): compañías × ejercicios. Dos de ellas necesitan 8 y 12 llamadas a `get_xbrl_fact`: **rompen el límite de 8 a propósito**.

En la primera pasada se corrió **A4 tal cual** para medir los fallos; el bloque siguiente usa el preset `final` actual (A6). Los resultados van a `resultados_dificil/` para no mezclarse con los del golden original. La predicción está escrita en `docs/predicciones.md` antes de ejecutar.


**v2 (23-sep, tras la primera pasada).** El golden pasa a 18 preguntas: `cifras_aceptables` en las multi-entidad (vale la magnitud derivada o el valor XBRL de la ganadora, que es lo que manda el prompt), `fuente` `xbrl` o `ninguna` en gY-010, y tres preguntas nuevas con la cifra **solo en texto** (Item 7A: gY-016, gY-017 y gY-018, esta última pensada para `read_section`). `puntuar(..., golden=...)` puntúa lo ya guardado con la versión actual. Detalle y predicciones en `docs/predicciones.md`.

In [2]:
from agente import interfaz
from agente.config import arquitectura

GOLDEN_DIFICIL = "data/golden_set_dificil.jsonl"
ARQ_DIFICIL    = arquitectura("final").nombre  # A6: mismo preset que evaluar()
REPS_DIFICIL   = 1                      # una primera pasada; sube a 3 si el patrón es claro

interfaz.CARPETA_RESULTADOS = "resultados_dificil"      # carpeta aparte
_golden_d = {g["id"]: g for g in interfaz.leer_golden(GOLDEN_DIFICIL)}
try:
    tablas_d = {}
    if HAY_CLAVE:
        for rep in range(1, REPS_DIFICIL + 1):
            print(f"\n== {ARQ_DIFICIL} · golden difícil · repetición {rep}/{REPS_DIFICIL} ==")
            ejecutar(GOLDEN_DIFICIL, ARQ_DIFICIL, rep)
            tablas_d[rep] = puntuar(ARQ_DIFICIL, rep, con_recall=False, golden=GOLDEN_DIFICIL)
            display(resumir(tablas_d[rep], f"{ARQ_DIFICIL} · difícil rep{rep}"))
    else:
        for carpeta in sorted(dir_arquitectura(ARQ_DIFICIL).glob("rep*")):
            if (carpeta / "crudo").is_dir() and any((carpeta / "crudo").glob("*.json")):
                tablas_d[int(carpeta.name[3:])] = puntuar(ARQ_DIFICIL, int(carpeta.name[3:]), con_recall=False, golden=GOLDEN_DIFICIL)

    def _colorear(fila):                       # autónomo: no depende del bloque 4
        return ["background-color:#fde2e2" if fila.get("acierto") is False else
                ("background-color:#e2f5e2" if fila.get("acierto") is True else "") for _ in fila]

    def _traza(arq, rep, id_):
        reg = json.loads((dir_rep(arq, rep) / "crudo" / f"{id_}.json").read_text(encoding="utf-8"))
        it = _golden_d.get(reg["item"]["id"], reg["item"])   # el golden ACTUAL (v2), no el guardado
        print("=" * 88)
        print(f"[{it['id']} · {it['familia']}] {it['pregunta']}")
        print(f"  esperado: {str(it.get('respuesta_esperada'))[:160]}")
        if it.get("nota"):
            print(f"  nota: {it['nota'][:160]}")
        print()
        print(reg.get("error") or pretty_trace(reg["resultado"]))

    if tablas_d:
        t = tablas_d[max(tablas_d)]
        cols = ["id","familia","acierto","honestidad","cifra","cita","trayectoria","fuente",
                "cifra_dada","cifra_esperada","ratio_cifra","n_llamadas","limite_alcanzado",
                "corrigio_cifra","reintentos_vacios","latencia_s","coste_usd"]
        cols = [c for c in cols if c in t]
        display(t[cols].style.apply(_colorear, axis=1).format({"latencia_s":"{:.1f}","coste_usd":"{:.4f}","ratio_cifra":"{:.3f}"}))
        display(comparar([ARQ_DIFICIL]))
        # las trazas de todos los fallos, para leer POR QUÉ
        for id_ in t.loc[t["acierto"] == False, "id"]:    # noqa: E712
            _traza(ARQ_DIFICIL, max(tablas_d), id_)
finally:
    interfaz.CARPETA_RESULTADOS = "resultados"           # dejarlo como estaba, pase lo que pase



== a4_comparativas · golden difícil · repetición 1/1 ==
  retrieval precargado: 1749 vectores, 1749 fragmentos + BM25 (0.3 s)
  [1/15] gY-001 … 11.5s · 0.79¢ · 2 llamadas · fuente=ninguna
  [2/15] gY-002 … 11.2s · 0.94¢ · 2 llamadas · fuente=ninguna
  [3/15] gY-003 … 52.8s · 1.80¢ · 4 llamadas · fuente=ninguna
  [4/15] gY-004 … 19.8s · 1.18¢ · 1 llamadas · fuente=ninguna
  [5/15] gY-005 … 26.4s · 0.89¢ · 2 llamadas · fuente=ninguna
  [6/15] gY-006 … 10.9s · 0.81¢ · 3 llamadas · fuente=xbrl
  [7/15] gY-007 … 14.6s · 1.18¢ · 7 llamadas · fuente=xbrl
  [8/15] gY-008 … 28.4s · 1.57¢ · 3 llamadas · fuente=xbrl
  [9/15] gY-009 … 48.4s · 1.60¢ · 6 llamadas · fuente=xbrl
  [10/15] gY-010 … 32.6s · 1.18¢ · 3 llamadas · fuente=ninguna
  [11/15] gY-011 … 47.1s · 3.03¢ · 10 llamadas · fuente=xbrl
  [12/15] gY-012 … 38.4s · 2.04¢ · 9 llamadas · fuente=ninguna
  [13/15] gY-013 … 68.6s · 3.48¢ · 8 llamadas · fuente=ambas
  [14/15] gY-014 … 98.3s · 2.24¢ · 7 llamadas · fuente=ambas
  [15/15] gY-015 …

{'versión': 'a4_comparativas · difícil rep1',
 'acierto': 0.4666666666666667,
 'cita': 1.0,
 'cifra': 0.2,
 'trayectoria': 1.0,
 'recall@5': nan,
 'coste medio (¢)': 1.65065,
 'latencia media (s)': 36.11034082658589,
 'llamadas/pregunta': 5.333333333333333,
 '% fuente=ninguna': 0.5333333333333333,
 'errores': 0,
 'reintentos proveedor': 0,
 'reintentos por respuesta vacía': 0,
 '% sin respuesta': 0.0,
 '% corrigió cifra': 0.0,
 '% corrigió cita': 0.0,
 'reintentos esquema': 0.3333333333333333,
 '% límite alcanzado': 0.2,
 '% búsquedas reescritas': np.float64(0.6),
 '% búsquedas con ticker': 1.0,
 'acierto numerica': nan,
 'acierto extractiva': nan,
 'acierto comparativa': nan,
 'acierto honestidad': 1.0,
 'acierto multi': 0.4,
 'acierto multi_temporal': 0.0,
 'honestidad': 0.8333333333333334}

,id,familia,acierto,honestidad,cifra,cita,trayectoria,fuente,cifra_dada,cifra_esperada,ratio_cifra,n_llamadas,limite_alcanzado,corrigio_cifra,reintentos_vacios,latencia_s,coste_usd
0,gY-001,honestidad,True,True,None,None,True,ninguna,nan,nan,nan,2,False,False,0,11.5,0.0079
1,gY-002,honestidad,True,True,None,None,None,ninguna,nan,nan,nan,2,False,False,0,11.2,0.0094
2,gY-003,honestidad,True,True,None,None,None,ninguna,nan,nan,nan,4,False,False,0,52.8,0.0180
3,gY-004,honestidad,True,True,None,None,None,ninguna,nan,nan,nan,1,False,False,0,19.8,0.0118
4,gY-005,honestidad,True,True,None,None,None,ninguna,nan,nan,nan,2,False,False,0,26.4,0.0089
5,gY-006,multi,True,None,True,None,True,xbrl,112010000000.000000,112010000000.000000,1.000,3,False,False,0,10.9,0.0081
6,gY-007,multi,True,None,True,None,True,xbrl,86810000000.000000,86810000000.000000,1.000,7,False,False,0,14.6,0.0118
7,gY-008,multi,False,None,False,None,True,xbrl,61087000000.000000,3715000000.000000,16.443,3,False,False,0,28.4,0.0157
8,gY-009,multi,False,None,False,None,True,xbrl,81453000000.000000,62.420000,1304918295.418,6,False,False,0,48.4,0.0160
9,gY-010,multi,False,False,False,None,True,ninguna,nan,57372000000.000000,nan,3,False,False,0,32.6,0.0118


,arquitectura,reps,acierto,cita,cifra,trayectoria,recall@5,coste medio (¢),latencia media (s),llamadas/pregunta,...,% límite alcanzado,% búsquedas reescritas,% búsquedas con ticker,acierto numerica,acierto extractiva,acierto comparativa,acierto honestidad,acierto multi,acierto multi_temporal,honestidad
0,a4_comparativas,1,0.466667,1.0,0.2,1.0,NaN,1.65065,36.110341,5.333333,...,0.2,0.6,1.0,NaN,NaN,NaN,1.0,0.4,0.0,0.833333


[gY-008 · multi] ¿Cuánto más gastó Alphabet que Meta en I+D en FY2025?
  esperado: Diferencia GOOGL − META = 3,715,000,000 USD. Convención: cifra=la diferencia, en USD.
  nota: Predicción: el verificador de cifras rechazará la diferencia (no es un hecho XBRL de ningún ticker) y forzará al modelo a poner el valor de una sola compañía.

  1. list_available()
       -> Contenido disponible en el corpus: - AAPL (Apple Inc.): ejercicios FY2024, FY2025; secciones disponibles: 1A, 7, 7A, 8. - AMZN (AMAZON COM INC): ejercicios FY2024, FY2025; secciones disponibles: 1A, 7, 7A, 8. - GOOGL (Al…
  2. get_xbrl_fact(ticker='GOOGL', fiscal_year=2025, concept='ResearchAndDevelopmentExpense')
  3. get_xbrl_fact(ticker='META', fiscal_year=2025, concept='ResearchAndDevelopmentExpense')
       -> GOOGL FY2025 · ResearchAndDevelopmentExpense = 61,087,000,000 USD (cierre de ejercicio 2025-12-31, según el 10-K)
       -> META FY2025 · ResearchAndDevelopmentExpense = 57,372,000,000 USD (cierre de ejercicio 20

## 7b · La escalera entera contra los dos golden: ejecución y reconciliación

La ejecución larga va **fuera del notebook**, en PowerShell, porque son ~75 preguntas y más de una hora:

```powershell
uv run python -m agente.lotes correr --plan     # qué falta, minutos y dólares; no gasta
uv run python -m agente.lotes correr            # toda la escalera, golden difícil, 1 rep
```

No repite nada de lo hecho: salta cada pregunta con respuesta guardada y solo repite las que no la dejaron (error del proveedor o turno vacío). Se puede cortar con Ctrl+C y volver a lanzar.

La celda de abajo **no repite preguntas del agente**: re-puntúa lo guardado con los evaluadores actuales, mide recall@5 en ambos golden sets y escribe las comparativas en `resultados/reconciliacion/`. Si falta una consulta reescrita en la caché, esa medición puede llamar al modelo.

In [ ]:
from agente import lotes

lado = lotes.reconciliar()          # repuntúa lo guardado y mide recall@5 en ambos golden
display(lotes.estado("dificil")[["arquitectura", "rep", "hechas", "faltan", "reparables", "obsoletas"]])
display(lado)


## 7c · Baseline frente a final en los dos golden sets

La tabla exigida para el informe se genera con `python -m agente.lotes reconciliar`, junto a las comparativas de **todas** las arquitecturas probadas. Usa el preset al que apunta `final` en `agente/config.py` (actualmente `a6_cifras_texto`) y lo enfrenta al baseline **por separado** en el golden original y en el difícil. Ejecuta primero el bloque 7b para actualizar los ficheros.

Cada tabla incluye acierto total y por familia, recall@5 del retriever, coste medio en céntimos, latencia media en segundos y llamadas a herramientas por pregunta. El mejor valor de cada columna se marca en negrita; en caso de empate se marcan ambos. Los porcentajes y promedios son medias de las repeticiones disponibles: 3 para el baseline original, 1 para A6 en el original y 1 para cada arquitectura en el difícil. El recall se calcula solo sobre preguntas con ancla (14 originales y 4 difíciles) y puede necesitar una llamada para reescribir una consulta si falta en la caché.

Las tablas listas para incorporar al informe quedan en `resultados/reconciliacion/baseline_vs_final_original.md` y `baseline_vs_final_dificil.md`, con versiones CSV numéricas al lado.


In [ ]:
# Tablas regeneradas por el bloque 7b; coste y latencia figuran como columnas.
from agente.config import arquitectura
print(f"Preset final: {arquitectura('final').nombre}")
for golden in lotes.GOLDENS:
    ruta = raiz_repo() / 'resultados' / 'reconciliacion' / f'baseline_vs_final_{golden}.md'
    display(Markdown(ruta.read_text(encoding='utf-8')))


## Siguiente peldaño

1. Cambia `ARQ` en el bloque 1: `a1_guardrails` → `a2_retrieval` → `a3_hibrido` → `a4_comparativas` → `a5_limites` → `a6_cifras_texto`.
2. Ejecuta los bloques 3, 4 y 5.
3. La comparativa crece una fila. Lo que empeore, también se cuenta.

Y al congelar el baseline, en la terminal:

```
git tag baseline-congelado
git add resultados && git commit -m "Baseline congelado: 3 repeticiones"
```